# 19.21 Uplift modeling

Uplift modeling predicts who changes because of an action, not who would act anyway.

The notebook simulates randomized-to-biased treatment assignment with known individual uplift. A T-learner estimates treated and control response curves, then a Qini-style gain checks whether high-scored users truly have larger incremental response.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1919
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _standardize(X):
    scaler = StandardScaler()
    return scaler.fit_transform(np.asarray(X, dtype=float))


def _binary_target(y):
    y = np.asarray(y)
    classes = np.unique(y)
    if len(classes) == 2:
        return (y == classes[-1]).astype(int)
    return (y == classes[-1]).astype(int)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def fit_logistic(X, y):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X, y)
    return clf


## Build uplift once on D1\n\nThe lesson formula is $$u(x)=P(Y=1\mid T=1,x)-P(Y=1\mid T=0,x).$$\nThe lesson components satisfy $0.300+0.180+0.120=0.600$ and the leading share is $0.300/0.600=0.500$.

In [ ]:

class ConstantProbabilityModel:
    def __init__(self, probability):
        self.probability = probability

    def predict_proba(self, X):
        p1 = np.full(X.shape[0], self.probability)
        p0 = 1.0 - p1
        return np.column_stack([p0, p1])


def uplift_score(model_t, model_c, X):
    treated_prob = model_t.predict_proba(X)[:, 1]
    control_prob = model_c.predict_proba(X)[:, 1]
    return treated_prob - control_prob


def qini_gain(T, Y, uplift, e):
    order = np.argsort(-uplift)
    T = np.asarray(T)[order]
    Y = np.asarray(Y)[order]
    e = np.clip(np.asarray(e)[order], 0.05, 0.95)
    transformed = Y * (T / e - (1.0 - T) / (1.0 - e))
    cumulative = np.cumsum(transformed)
    random_line = np.linspace(cumulative[0], cumulative[-1], len(cumulative))
    return float(np.trapz(cumulative - random_line) / len(cumulative))

components = np.array([0.300, 0.180, 0.120])
total = float(components.sum())
share = float(abs(components[0]) / np.abs(components).sum())
toy_X = np.ones((2, 1))
toy_uplift = uplift_score(ConstantProbabilityModel(0.9), ConstantProbabilityModel(0.3), toy_X)[0]
assert round(total, 3) == 0.600
assert round(share, 3) == 0.500
assert round(float(toy_uplift), 3) == 0.600
print("toy uplift", round(float(toy_uplift), 3))
print("leading component share", round(share, 3))


Now create outcomes where the ground-truth uplift varies by features; the assignment policy becomes more biased along the ladder.

In [ ]:

def make_uplift_dataset(X, y, assignment_bias, seed):
    local_rng = np.random.default_rng(seed)
    Xs = _standardize(X)
    yb = _binary_target(y)
    z = Xs[:, 0]
    e = np.clip(sigmoid(assignment_bias * z), 0.05, 0.95)
    T = local_rng.binomial(1, e)
    if T.sum() == 0 or T.sum() == len(T):
        T = np.arange(len(T)) % 2
    base_prob = np.clip(sigmoid(-0.4 + 0.6 * z + 0.4 * yb), 0.02, 0.82)
    true_uplift = np.clip(0.06 + 0.24 * sigmoid(Xs[:, 0]), -0.10, 0.30)
    p_control = base_prob
    p_treated = np.clip(base_prob + true_uplift, 0.01, 0.99)
    p_observed = np.where(T == 1, p_treated, p_control)
    Y = local_rng.binomial(1, p_observed)
    return {
        "X": Xs[:, : min(6, Xs.shape[1])],
        "T": T,
        "Y": Y,
        "e": e,
        "true_uplift": true_uplift,
    }

first = make_uplift_dataset(clf_ladder()[0][1], clf_ladder()[0][2], 0.0, SEED)
print("mean true uplift", round(float(first["true_uplift"].mean()), 3))
print("assignment range", round(float(first["e"].min()), 3), round(float(first["e"].max()), 3))


## Dataset ladder

The base F15 ladder supplies features; treatment assignment moves from randomized to biased.

In [ ]:

uplift_specs = [
    ("D1 randomized toy", 0.0),
    ("D2 mild targeting bias", 0.5),
    ("D3 tabular policy bias", 1.0),
    ("D4 image-feature policy", 1.5),
    ("D5 biased historical targeting", 2.2),
]
uplift_rungs = []
for idx, ((base_name, X, y), (name, assignment_bias)) in enumerate(zip(clf_ladder(), uplift_specs)):
    data = make_uplift_dataset(X, y, assignment_bias, SEED + idx)
    uplift_rungs.append((name, data, assignment_bias))
    print(name, data["X"].shape, "treated", int(data["T"].sum()), "control", int((1 - data["T"]).sum()))
    print("sample uplift", np.round(data["true_uplift"][:3], 3))


## Run a T-learner uplift model across D1-D5

In [ ]:

def fit_outcome_model(X, Y):
    if len(np.unique(Y)) < 2:
        return ConstantProbabilityModel(float(np.mean(Y)))
    model = LogisticRegression(max_iter=2000)
    model.fit(X, Y)
    return model


def fit_t_learner(data):
    treated = data["T"] == 1
    control = data["T"] == 0
    model_t = fit_outcome_model(data["X"][treated], data["Y"][treated])
    model_c = fit_outcome_model(data["X"][control], data["Y"][control])
    return model_t, model_c

uplift_results = []
for name, data, assignment_bias in uplift_rungs:
    model_t, model_c = fit_t_learner(data)
    pred_uplift = uplift_score(model_t, model_c, data["X"])
    gain = qini_gain(data["T"], data["Y"], pred_uplift, data["e"])
    rank_corr = float(np.corrcoef(pred_uplift, data["true_uplift"])[0, 1])
    top_mask = pred_uplift >= np.percentile(pred_uplift, 70)
    top_true = float(data["true_uplift"][top_mask].mean())
    uplift_results.append((name, gain, rank_corr, top_true, pred_uplift))

print("rung | qini_gain | corr_with_true_uplift | top30_true_uplift")
for row in uplift_results:
    print(row[0], *(round(float(v), 3) for v in row[1:4]))


## Results visualization

Left: uplift bins with true incremental response. Right: Qini gain as assignment bias rises.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
for ax, (row, rung) in zip(axes[:5], zip(uplift_results, uplift_rungs)):
    name, gain, rank_corr, top_true, pred_uplift = row
    data = rung[1]
    bins = np.quantile(pred_uplift, [0.0, 0.33, 0.66, 1.0])
    labels = []
    values = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (pred_uplift >= lo) & (pred_uplift <= hi)
        labels.append(f"{lo:.2f}\n{hi:.2f}")
        values.append(float(data["true_uplift"][mask].mean()))
    ax.bar(labels, values, color="seagreen")
    ax.set_title(name)
    ax.set_ylabel("true uplift")
axes[5].plot(range(1, 6), [row[1] for row in uplift_results], marker="o")
axes[5].set_xticks(range(1, 6))
axes[5].set_xlabel("rung")
axes[5].set_ylabel("Qini gain")
axes[5].set_title("Gain vs assignment bias")
plt.tight_layout()


## Pitfall on D5: training only on treated outcomes

The wrong model ranks response among treated users, not incremental response. The fix compares treated and control outcome models and uses propensity-aware evaluation.

In [ ]:

name, data, assignment_bias = uplift_rungs[-1]
response_model = fit_outcome_model(data["X"][data["T"] == 1], data["Y"][data["T"] == 1])
wrong_score = response_model.predict_proba(data["X"])[:, 1]
model_t, model_c = fit_t_learner(data)
fixed_score = uplift_score(model_t, model_c, data["X"])
wrong_gain = qini_gain(data["T"], data["Y"], wrong_score, data["e"])
fixed_gain = qini_gain(data["T"], data["Y"], fixed_score, data["e"])
print("wrong response-model gain", round(wrong_gain, 3))
print("fixed T-learner gain", round(fixed_gain, 3))
print("negative uplift share", round(float(np.mean(fixed_score < 0.0)), 3))


## Evaluate it + Practice

- Metric: Qini gain and correlation with known simulated uplift
- No-skill baseline: rank by treated response probability only
- Cheap sanity check: randomized D1 should keep treatment/control counts balanced
- Ablation: collapse to a response model and Qini gain should degrade
- Failure signals: high response but low or negative incremental effect, or no control support in a bin

Practice prompts:
1. Change the D5 stress level and predict which metric should move first.

2. Add one subgroup or environment slice and explain whether the conclusion changes.

3. Replace the default logistic model with another CPU-only sklearn classifier and compare the same metric.